# MAMoE-50 Kaggle Training Pipeline

Notebook ini dirancang khusus untuk melatih arsitektur **MAMoE-50** pada *Kaggle GPU* (T4 x2 atau P100). Notebook ini akan meng-clone repositori GitHub utama, memuat dataset VQFat/T5Gemma dari `/kaggle/input`, dan mengeksekusi JAX training loop.

In [ ]:
# 1. Setup Environment (Instalasi Dependensi JAX Khusus GPU)
!pip install --upgrade jax jaxlib flax optax -q
!nvidia-smi

In [ ]:
# 2. Clone Repositori MAMoE-50 dari GitHub
# Ganti URL ini dengan link repositori GitHub Anda jika private, gunakan Personal Access Token (PAT)
!git clone https://github.com/USERNAME/MAMoE-50.git
%cd MAMoE-50

In [ ]:
# 3. Verifikasi Path Dataset VQFat dari Kaggle Input
import os
dataset_path = '/kaggle/input/vqfat-indonesian-corpus/train_chunks.npy'

if os.path.exists(dataset_path):
    print("✅ Dataset VQFat Ditemukan!")
else:
    print("⚠️ DATASET TIDAK DITEMUKAN. Pastikan Anda telah menekan 'Add Data' dan memasukkan VQFat dataset.")

In [ ]:
# 4. Update Path Dataset pada train.py dan Atur Hiperparameter
# Kita akan menggunakan metode sed untuk me-replace path default di train.py agar mengarah ke kaggle input
!sed -i "s|'data/pretrain/train_chunks.npy'|'/kaggle/input/vqfat-indonesian-corpus/train_chunks.npy'|g" train.py

# Kembalikan batch_size dan seq_len ke ukuran produksi Kaggle GPU
!sed -i 's/batch_size = 2/batch_size = 16/g' train.py
!sed -i 's/seq_len = 64/seq_len = 1024/g' train.py

In [ ]:
# 5. Mulai Pelatihan!
# Script train.py kini menggunakan XLA JIT murni yang akan memaksimalkan utilitas P100/T4.
!python train.py